# FMR — Faithfulness-LoRA Training & Evaluation (Kaggle T4 x2)

Runs QLoRA fine-tuning and evaluation on **Kaggle** with full token preservation (`MAX_LEN=1024`) and pushes results to GitHub.

**Prerequisites:** Add `HF_TOKEN` and `GH_TOKEN` to **Add-ons -> Secrets** with the checkboxes checked.

In [ ]:
# 0. GPU + Memory Config
import os, sys, shutil, gc, json, traceback
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# 1. Load Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ['HF_TOKEN'] = user_secrets.get_secret('HF_TOKEN')
    os.environ['GH_TOKEN'] = user_secrets.get_secret('GH_TOKEN')
    print('Loaded tokens from Kaggle Secrets successfully.')
except Exception as e:
    print('ERROR: Kaggle secrets not found. Please check Add-ons -> Secrets.')
    print(e)

# 2. Setup Working Directory & Clone Repo
if os.path.exists('/kaggle/working'):
    os.chdir('/kaggle/working')

if not os.path.exists('fmr-thesis'):
    print('Cloning repository...')
    gh_token = os.environ.get('GH_TOKEN', '')
    clone_cmd = f"git clone -b master https://{gh_token}@github.com/Ankit-blip737/fmr-thesis.git"
    os.system(clone_cmd)

%cd /kaggle/working/fmr-thesis
!git pull origin master --rebase
!git config --global user.email 'kaggle@fmr.run'
!git config --global user.name 'FMR Kaggle'

# 3. Install Dependencies
!pip -q install 'transformers>=4.49.0' peft bitsandbytes accelerate datasets qwen-vl-utils pillow scikit-learn pyyaml
!pip -q install -e fmr[real]

import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
    print('Total VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')

## 2. Prepare Distillation Dataset (using 4-bit VLM)

In [ ]:
%cd /kaggle/working/fmr-thesis
import sys, json, os, gc, traceback
import numpy as np
import torch
from PIL import Image

sys.path.insert(0, '/kaggle/working/fmr-thesis/fmr/src')
from huggingface_hub import login
if os.environ.get('HF_TOKEN'):
    login(os.environ['HF_TOKEN'])

from fmr.types import Sample, VLMOutput
from fmr.faithfulness.decompose import decompose_rationale
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

# ---- hf_hub patch ----
def patch_config_for_strict_hub():
    try:
        from transformers import configuration_utils
        BOOL = frozenset({'use_cache','output_attentions','output_hidden_states',
                          'return_dict','tie_word_embeddings','is_decoder',
                          'add_cross_attention','chunk_size_feed_forward'})
        def _san(d):
            for k,v in list(d.items()):
                if k in BOOL and v is None: d[k] = True
                elif isinstance(v, dict): _san(v)
        orig = configuration_utils.PretrainedConfig.from_dict.__func__
        if not getattr(orig, '_fmr_patched', False):
            orig._fmr_patched = True
            def patched(cls, cfg, **kw): _san(cfg); return orig(cls, cfg, **kw)
            configuration_utils.PretrainedConfig.from_dict = classmethod(patched)
    except Exception as e: print('patch skipped:', e)
patch_config_for_strict_hub()

BASE_MODEL = 'Qwen/Qwen2.5-VL-3B-Instruct'
_bnb4 = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                            bnb_4bit_compute_dtype=torch.bfloat16)

def _blank_like(img): return Image.new('RGB', img.size, (127,127,127))

class RealAnswerVLM:
    is_reasoning = True
    def __init__(self, model_id=BASE_MODEL, max_new_tokens=64, bnb=None):
        self.name = model_id; self.max_new_tokens = max_new_tokens
        self.processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
        self.model = AutoModelForImageTextToText.from_pretrained(
            model_id,
            quantization_config=bnb or _bnb4,
            device_map='auto', trust_remote_code=True).eval()
    def _img(self, s, v):
        i = s.meta['_pil']
        return i if v=='original' else (_blank_like(i) if v=='blank' else s.meta.get('_pil_other', _blank_like(i)))
    def _msgs(self, q, ch):
        return [{'role':'user','content':[{'type':'image'},{'type':'text','text':f'{q}\nAnswer with one of: {", ".join(ch)}.\nAnswer:'}]}]
    @torch.no_grad()
    def _clp(self, img, q, ch, c):
        p = self.processor.apply_chat_template(self._msgs(q,ch),tokenize=False,add_generation_prompt=True)
        f = p + ' ' + c
        ep = self.processor(text=[p], images=[img], return_tensors='pt').to(self.model.device)
        ef = self.processor(text=[f], images=[img], return_tensors='pt').to(self.model.device)
        n = ep['input_ids'].shape[1]
        lg = self.model(**ef).logits[0].float().log_softmax(-1)
        ids = ef['input_ids'][0]
        return float(sum(lg[t-1, ids[t]].item() for t in range(n, ids.shape[0])))
    @torch.no_grad()
    def _dist(self, img, q, ch):
        l = np.array([self._clp(img,q,ch,c) for c in ch])
        if not np.all(np.isfinite(l)): return np.full(len(ch), 1.0/len(ch))
        l -= l.max(); p = np.exp(l); return p / p.sum()
    @torch.no_grad()
    def _rat(self, img, q, ch):
        m = [{'role':'user','content':[{'type':'image'},{'type':'text','text':f'{q}\nReason step by step, then answer with one of: {", ".join(ch)}.'}]}]
        pr = self.processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
        e = self.processor(text=[pr], images=[img], return_tensors='pt').to(self.model.device)
        g = self.model.generate(**e, max_new_tokens=self.max_new_tokens, do_sample=False)
        return self.processor.batch_decode(g[:, e['input_ids'].shape[1]:], skip_special_tokens=True)[0]
    def generate(self, s, variant='original', temperature=0.0, draw=0):
        ch = s.answer_choices or [s.answer]; img = self._img(s, variant)
        p = self._dist(img, s.question, ch)
        steps = decompose_rationale(self._rat(img, s.question, ch)) if variant=='original' else []
        return VLMOutput(sample_id=s.sample_id, answer=ch[int(np.argmax(p))],
                         steps=steps, answer_logits=p, variant=variant, meta={'model':self.name})

# ---- Load subset of VQA-RAD for distillation ----
from datasets import load_dataset
N_TRAIN = 100   # Scaled for Kaggle T4x2
ds = load_dataset('flaviagiammarino/vqa-rad', split='train')
rows = [r for r in ds if r['answer'].strip().lower() in {'yes','no'}][:N_TRAIN]
samples = []
for i,r in enumerate(rows):
    s = Sample(sample_id=f'vqarad-tr-{i:03d}', question=r['question'],
               answer=r['answer'].strip().lower(), modality='xray', answer_choices=['yes','no'])
    s.meta['_pil'] = r['image'].convert('RGB')
    s.meta['_pil_other'] = rows[(i+7)%len(rows)]['image'].convert('RGB')
    samples.append(s)

from fmr.training import FaithfulnessLoRAConfig
from fmr.training.faithfulness_lora import DistillExample
from fmr.correction import CorrectionConfig, correct_sample

# Data-prep VLM
vlm = RealAnswerVLM(BASE_MODEL, max_new_tokens=64)
cfg = FaithfulnessLoRAConfig(correction=CorrectionConfig(n_probes=2, vcd_margin=1.0))
os.makedirs('fmr/results', exist_ok=True)

print(f'Scoring and extracting verified distill targets from {len(samples)} samples...')
res = [correct_sample(vlm, s, config=cfg.correction) for s in samples]
fs_after = sorted(r.fs_after for r in res)
thr = fs_after[len(fs_after)//2] if fs_after else 0.0
distill = [DistillExample(sample_id=r.sample_id, question=s.question,
                           target_answer=r.corrected.answer,
                           target_rationale=r.corrected.rationale,
                           fs=r.fs_after, source_correction_applied=r.applied)
            for r,s in zip(res,samples)
            if r.fs_after >= thr and r.corrected.rationale.strip()]
print(f'{len(distill)} distill targets (fs_after>=median={thr:.3f}) from {len(samples)} samples')
json.dump([d.__dict__ for d in distill],
          open('fmr/results/faithfulness_lora_distill_set.json','w'), indent=2)

# ---- FREE data-prep VLM before LoRA training ----
_proc_ref = vlm.processor
del vlm.model, vlm
gc.collect(); torch.cuda.empty_cache()
print('VRAM after cleanup:', round(torch.cuda.memory_allocated()/1e9,2), 'GB allocated')

## 3. QLoRA Training & Evaluation

In [ ]:
%cd /kaggle/working/fmr-thesis
report = {'status': 'started', 'n_distill': len(distill)}
try:
    assert len(distill) >= 4, f'too few distill targets ({len(distill)}) to train'
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from transformers import BitsAndBytesConfig, TrainingArguments, Trainer

    proc = _proc_ref
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                             bnb_4bit_compute_dtype=torch.bfloat16)
    base = AutoModelForImageTextToText.from_pretrained(
        cfg.base_model, quantization_config=bnb,
        device_map='auto', trust_remote_code=True)
    base.config.use_cache = False
    base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)
    lora_cfg = LoraConfig(r=cfg.lora_r, lora_alpha=cfg.lora_alpha,
                          lora_dropout=cfg.lora_dropout,
                          target_modules=['q_proj','k_proj','v_proj','o_proj'],
                          task_type='CAUSAL_LM')
    model = get_peft_model(base, lora_cfg)
    model.print_trainable_parameters()

    sid2img = {s.sample_id: s.meta['_pil'] for s in samples}
    sid2q   = {s.sample_id: s.question      for s in samples}
    pad_id  = proc.tokenizer.pad_token_id or proc.tokenizer.eos_token_id
    _cfg    = getattr(base, 'config', None)
    img_tok = getattr(_cfg, 'image_token_id', None) or getattr(_cfg, 'image_token_index', None)

    MAX_LEN = 1024   # Preserves full image tokens!

    def collate(batch):
        texts, images = [], []
        for ex in batch:
            msgs = [
                {'role':'user','content':[{'type':'image'},{'type':'text','text':sid2q[ex['sample_id']]}]},
                {'role':'assistant','content':[{'type':'text','text':ex['target_rationale']+'\nAnswer: '+ex['target_answer']}]}
            ]
            texts.append(proc.apply_chat_template(msgs, tokenize=False))
            images.append(sid2img[ex['sample_id']])
        enc = proc(text=texts, images=images, return_tensors='pt', padding=True,
                   truncation=True, max_length=MAX_LEN)
        labels = enc['input_ids'].clone()
        labels[labels == pad_id] = -100
        if img_tok is not None: labels[labels == img_tok] = -100
        enc['labels'] = labels
        return enc

    train_args = TrainingArguments(
        output_dir='/kaggle/working/flora',
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_train_epochs=cfg.epochs,
        learning_rate=cfg.learning_rate,
        bf16=True,
        fp16=False,
        logging_steps=2,
        report_to=[],
        remove_unused_columns=False,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
        dataloader_pin_memory=False,
    )
    Trainer(model=model, args=train_args,
            train_dataset=[d.__dict__ for d in distill],
            data_collator=collate).train()
    model.save_pretrained('/kaggle/working/flora/adapter')
    report['training'] = 'ok'

    # ---- Eval: frozen-base vs LoRA ----
    from fmr.correction import correct_sample
    dse = load_dataset('flaviagiammarino/vqa-rad', split='test')
    er  = [r for r in dse if r['answer'].strip().lower() in {'yes','no'}][:50]
    eval_s = []
    for i,r in enumerate(er):
        s = Sample(sample_id=f'vqarad-te-{i:03d}', question=r['question'],
                   answer=r['answer'].strip().lower(), modality='xray',
                   answer_choices=['yes','no'])
        s.meta['_pil'] = r['image'].convert('RGB')
        s.meta['_pil_other'] = er[(i+7)%len(er)]['image'].convert('RGB')
        eval_s.append(s)

    def eval_model(m, tag):
        w = RealAnswerVLM.__new__(RealAnswerVLM)
        w.name = tag; w.max_new_tokens = 64; w.processor = proc; w.model = m.eval()
        gt = {s.sample_id: s.answer for s in eval_s}
        fs_list, ok_list = [], []
        for s in eval_s:
            r = correct_sample(w, s, config=CorrectionConfig(vcd_margin=1.0))
            fs_list.append(r.fs_after); ok_list.append(r.corrected.answer == gt[s.sample_id])
        return {'tag': tag, 'acc': float(np.mean(ok_list)), 'fs_mean': float(np.mean(fs_list))}

    # Frozen: disable adapters
    model.disable_adapter_layers()
    frozen = eval_model(model, 'frozen-base')

    # LoRA: enable adapters
    model.enable_adapter_layers()
    lora_ev = eval_model(model, 'faithfulness-lora')

    report.update({
        'status': 'ok', 'frozen': frozen, 'lora': lora_ev, 'n_eval': len(eval_s),
        'verdict': ('LoRA helped'
                    if lora_ev['fs_mean'] > frozen['fs_mean'] and lora_ev['acc'] >= frozen['acc']-0.02
                    else 'frozen retained (honest negative)')
    })

except Exception as e:
    report.update({'status': 'failed', 'error': str(e), 'traceback': traceback.format_exc()[-2000:]})
    print('LoRA run failed (captured):', e)

json.dump(report, open('fmr/results/faithfulness_lora_ablation.json','w'), indent=2)
print(json.dumps({k:v for k,v in report.items() if k!='traceback'}, indent=2))

## 4. Push Results to GitHub

In [ ]:
%cd /kaggle/working/fmr-thesis
!git add fmr/results/faithfulness_lora_*.json
!git commit -m '[B] Faithfulness-LoRA v3 results (Kaggle T4x2 GPU run)' || echo 'nothing to commit'
from fmr.scripts.push_results import push
push('fmr/results', '[B] Faithfulness-LoRA v3 results (Kaggle T4x2 GPU run)')